<a href="https://colab.research.google.com/github/arsverma5/ml2-final-project/blob/main/notebooks/neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/311_Service_Requests.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Data Prep

In [4]:
# load data
use_cols = [
    "Created Date",
    "Closed Date",
    "Agency",
    "Problem (formerly Complaint Type)",
    "Problem Detail (formerly Descriptor)",
    "Location Type",
    "Incident Zip",
    "Police Precinct",
    "Borough"
]

df = pd.read_csv(path, usecols=use_cols)
df.head()

/tmp/ipykernel_11595/1938478843.py:14: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=use_cols)


,Created Date,Closed Date,Agency,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Location Type,Incident Zip,Police Precinct,Borough
0,04/06/2026 02:04:20 AM,NaN,NYPD,Noise - Residential,Banging/Pounding,Residential Building/House,10040.0,Precinct 34,MANHATTAN
1,04/06/2026 02:04:10 AM,NaN,NYPD,Noise - Residential,Loud Television,Residential Building/House,10035.0,Precinct 25,MANHATTAN
2,04/06/2026 01:59:31 AM,NaN,NYPD,Noise - Commercial,Loud Music/Party,Club/Bar/Restaurant,11220.0,Precinct 72,BROOKLYN
3,04/06/2026 01:59:06 AM,NaN,NYPD,Noise - Residential,Loud Music/Party,Residential Building/House,11236.0,Precinct 69,BROOKLYN
4,04/06/2026 01:58:25 AM,NaN,DOHMH,Rodent,Condition Attracting Rodents,3+ Family Apt. Building,11233.0,Precinct 81,BROOKLYN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20741013 entries, 0 to 20741012
Data columns (total 9 columns):
 #   Column                                Dtype 
---  ------                                ----- 
 0   Created Date                          object
 1   Closed Date                           object
 2   Agency                                object
 3   Problem (formerly Complaint Type)     object
 4   Problem Detail (formerly Descriptor)  object
 5   Location Type                         object
 6   Incident Zip                          object
 7   Police Precinct                       object
 8   Borough                               object
dtypes: object(9)
memory usage: 1.4+ GB


In [6]:
print(df['Created Date'].min())
print(df['Created Date'].max())

01/01/2020 01:00:00 AM
12/31/2025 12:59:58 AM


In [7]:
df = df.rename(columns={
    'Problem (formerly Complaint Type)': 'complaint_type',
    'Created Date': 'created_date',
    'Closed Date': 'closed_date'
})

df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

df['complaint_type'] = df['complaint_type'].str.strip().str.lower()

print(df['complaint_type'].value_counts())

complaint_type
illegal parking               2636418
noise - residential           2380386
heat/hot water                1613064
noise - street/sidewalk       1049279
blocked driveway               990664
                               ...   
forensic engineering                1
trapping pigeon                     1
sustainability enforcement          1
stalled sites                       1
quality of life                     1
Name: count, Length: 257, dtype: int64


In [8]:
# filter df to top 10
top10 = df['complaint_type'].value_counts().head(10)
print(top10)

complaint_type
illegal parking                        2636418
noise - residential                    2380386
heat/hot water                         1613064
noise - street/sidewalk                1049279
blocked driveway                        990664
unsanitary condition                    639365
request large bulky item collection     632148
street condition                        453472
plumbing                                389736
water system                            386697
Name: count, dtype: int64


In [10]:
df = df[df['complaint_type'].isin(top10.index)]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11171229 entries, 0 to 20741003
Data columns (total 9 columns):
 #   Column                                Dtype         
---  ------                                -----         
 0   created_date                          datetime64[ns]
 1   closed_date                           datetime64[ns]
 2   Agency                                object        
 3   complaint_type                        object        
 4   Problem Detail (formerly Descriptor)  object        
 5   Location Type                         object        
 6   Incident Zip                          object        
 7   Police Precinct                       object        
 8   Borough                               object        
dtypes: datetime64[ns](2), object(7)
memory usage: 852.3+ MB


In [11]:
# calculate resolution hours
df['resolution_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600
df = df.dropna(subset=['resolution_hours'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11103414 entries, 13 to 20741003
Data columns (total 10 columns):
 #   Column                                Dtype         
---  ------                                -----         
 0   created_date                          datetime64[ns]
 1   closed_date                           datetime64[ns]
 2   Agency                                object        
 3   complaint_type                        object        
 4   Problem Detail (formerly Descriptor)  object        
 5   Location Type                         object        
 6   Incident Zip                          object        
 7   Police Precinct                       object        
 8   Borough                               object        
 9   resolution_hours                      float64       
dtypes: datetime64[ns](2), float64(1), object(7)
memory usage: 931.8+ MB


In [12]:
print(f"Mean Resolution Time: {df['resolution_hours'].mean()}")

Mean Resolution Time: 76.07541527350158
